# 身份验证和授权
FastAPI-MCP 支持使用您现有的 FastAPI 依赖项进行身份验证和授权。

它还支持完整的 OAuth 2 流程，符合MCP Spec 2025-03-26。

值得注意的是，大多数 MCP 客户端目前不支持最新的 MCP 规范，因此在我们的示例中，我们可能会使用桥接客户端，例如npx mcp-remote。我们建议您也使用它，并且我们将使用它来展示我们的示例。

​



## 基本令牌传递
如果您只是希望能够传递有效的授权标头，而不需要支持完整的身份验证流程，则无需执行任何特殊操作。

您只需要确保您的 MCP 客户端正在发送它：

```json
{
  "mcpServers": {
    "remote-example": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "http://localhost:8000/mcp",
        "--header",
        "Authorization:${AUTH_HEADER}"
      ]
    },
    "env": {
      "AUTH_HEADER": "Bearer <your-token>"
    }
  }
}
```

这足以将授权标头传递到您的 FastAPI 端点。

或者，如果您希望 MCP 服务器拒绝没有授权标头的请求，您可以添加依赖项：

本示例展示了如何拒绝任何未在授权标头中传递有效令牌的请求。 为了配置 auth 标头，MCP 服务器的配置文件应如下所示：

```json
{
  "mcpServers": {
    "remote-example": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "http://localhost:8000/mcp",
        "--header",
        "Authorization:${AUTH_HEADER}"
      ]
    },
    "env": {
      "AUTH_HEADER": "Bearer <your-token>"
    }
  }
}
```

In [ ]:

from examples.shared.apps.items import app # The FastAPI app
from examples.shared.setup import setup_logging

from fastapi import Depends
from fastapi.security import HTTPBearer

from fastapi_mcp import FastApiMCP, AuthConfig

setup_logging()

# Scheme for the Authorization header
token_auth_scheme = HTTPBearer()

# Create a private endpoint
@app.get("/private")
async def private(token = Depends(token_auth_scheme)):
    return token.credentials

# Create the MCP server with the token auth scheme
mcp = FastApiMCP(
    app,
    name="Protected MCP",
    auth_config=AuthConfig(
        dependencies=[Depends(token_auth_scheme)],
    ),
)

# Mount the MCP server
mcp.mount()


if __name__ == "__main__":
    import uvicorn
    
    uvicorn.run(app, host="0.0.0.0", port=8000)

## OAuth 流程
FastAPI-MCP 支持完整的 OAuth 2 流程，符合MCP Spec 2025-03-26。

它看起来是这样的：


In [ ]:
from fastapi import Depends
from fastapi_mcp import FastApiMCP, AuthConfig

mcp = FastApiMCP(
    app,
    name="MCP With OAuth",
    auth_config=AuthConfig(
        issuer=f"https://auth.example.com/",
        authorize_url=f"https://auth.example.com/authorize",
        oauth_metadata_url=f"https://auth.example.com/.well-known/oauth-authorization-server",
        audience="my-audience",
        client_id="my-client-id",
        client_secret="my-client-secret",
        dependencies=[Depends(verify_auth)],
        setup_proxies=True,
    ),
)

mcp.mount()

你可以这样调用它：

```json

{
  "mcpServers": {
    "fastapi-mcp": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "http://localhost:8000/mcp",
        "8080"  // Optional port number. Necessary if you want your OAuth to work and you don't have dynamic client registration.
      ]
    }
  }
}
```

您可以将其与任何支持 OAuth 2 规范的 OAuth 提供商一起使用。有关更多详细信息，请参阅AuthConfig的说明。

## 自定义 OAuth 元数据
如果您已经拥有一个正确配置的 OAuth 服务器，可以与 MCP 客户端配合使用，或者您希望完全控制元数据，则您可以直接提供自己的 OAuth 元数据：


In [ ]:
from fastapi import Depends
from fastapi_mcp import FastApiMCP, AuthConfig

mcp = FastApiMCP(
    app,
    name="MCP With Custom OAuth",
    auth_config=AuthConfig(
        # Provide your own complete OAuth metadata
        custom_oauth_metadata={
            "issuer": "https://auth.example.com",
            "authorization_endpoint": "https://auth.example.com/authorize",
            "token_endpoint": "https://auth.example.com/token",
            "registration_endpoint": "https://auth.example.com/register",
            "scopes_supported": ["openid", "profile", "email"],
            "response_types_supported": ["code"],
            "grant_types_supported": ["authorization_code"],
            "token_endpoint_auth_methods_supported": ["none"],
            "code_challenge_methods_supported": ["S256"]
        },

        # Your auth checking dependency
        dependencies=[Depends(verify_auth)],
    ),
)

mcp.mount()

此方法可让您完全控制 OAuth 元数据，并且在以下情况下很有用：

- 您已配置完全符合 MCP 标准的 OAuth 服务器
- 您需要自定义 OAuth 流程，超越代理方法提供的功能
- 您正在使用自定义或专门的 OAuth 实现

为了使其正常工作，您必须确保 mcp-remote在固定端口上运行（例如8080），然后http://127.0.0.1:8080/oauth/callback在您的 OAuth 提供程序中配置回调 URL。

## 使用 Auth0 的工作示例
有关 OAuth 与 Auth0 集成的完整工作示例，请查看示例文件夹中的Auth0 示例。此示例演示了使用 Auth0 作为 OAuth 提供程序的简单案例，并提供了 OAuth 流程的工作示例。

为了使其工作，您需要在项目根目录中有一个包含以下变量的 .env 文件：

```shell
AUTH0_DOMAIN=your-tenant.auth0.com
AUTH0_AUDIENCE=https://your-tenant.auth0.com/api/v2/
AUTH0_CLIENT_ID=your-client-id
AUTH0_CLIENT_SECRET=your-client-secret
```

您还需要确保在 Auth0 仪表板中正确配置回调 URL。

## AuthConfig 配置 

setup_proxies=True

大多数 OAuth 提供商需要进行一些调整才能与 MCP 客户端配合使用。这就是 setup_proxies=True 的作用所在--它会创建代理端点，使你的 OAuth 提供程序与 MCP 客户端兼容：



In [ ]:
mcp = FastApiMCP(
    app,
    auth_config=AuthConfig(
        # Your OAuth provider information
        issuer="https://auth.example.com",
        authorize_url="https://auth.example.com/authorize",
        oauth_metadata_url="https://auth.example.com/.well-known/oauth-authorization-server",

        # Credentials registered with your OAuth provider
        client_id="your-client-id",
        client_secret="your-client-secret",

        # Recommended, since some clients don't specify them
        audience="your-api-audience",
        default_scope="openid profile email",

        # Your auth checking dependency
        dependencies=[Depends(verify_auth)],

        # Create compatibility proxies - usually needed!
        setup_proxies=True,
    ),
)

您还需要确保在 OAuth 提供程序中正确配置回调 URL。例如，使用 mcp-remote 时，您必须使用固定端口。

## 为什么要使用代理？
代理解决了几个问题：

- 缺少注册端点：     
MCP 规范要求 OAuth 提供商支持动态客户端注册 (RFC 7591)，但许多提供商并未支持。
此外，动态客户端注册对于大多数用例来说可能有些过度。
此setup_fake_dynamic_registration选项（默认为 True）可创建一个兼容的端点，仅返回静态客户端 ID 和密钥。

- 范围处理：       
一些 MCP 客户端没有正确请求范围，因此我们的代理会为您添加必要的范围。

- 受众要求：    
某些 OAuth 提供商需要受众参数，而 MCP 客户端通常不会提供该参数。代理会自动添加此参数。

​


## 向 mcp-remote 添加固定端口

```json
{
  "mcpServers": {
    "example": {
      "command": "npx",
      "args": [
        "mcp-remote",
        "http://localhost:8000/mcp",
        "8080"
      ]
    }
  }
}
```

通常，mcp-remote 将在随机端口上启动，从而无法正确配置 OAuth 提供程序的回调 URL。

您必须确保 mcp-remote 在固定端口上运行，例如8080，然后http://127.0.0.1:8080/oauth/callback在您的 OAuth 提供程序中配置回调 URL。